# Group 4
Authors:  Yaroslav Kuzmin, Yevhen Nezghovorov, Hlieb Nikoniuk, Tymur Tleuberlin



In [5]:
import pandas as pd
import numpy as np
from statsmodels.tsa.ar_model import AutoReg
from statsmodels.tsa.vector_ar.var_model import VAR

import warnings
warnings.filterwarnings('ignore')

In [6]:
ovdp_url = "https://raw.githubusercontent.com/Tleuberlin/group4-project2-domain.github.io/main/datasets/ovdg_timeseries_5Y.csv"
ovdp = pd.read_csv(ovdp_url)
ovdp["Time"] = pd.to_datetime(ovdp["Time"])
ovdp_ts = ovdp.set_index("Time")["ОВДП"]

best_lag = None
best_aic = np.inf
for lag in range(1, 13):
    model = AutoReg(ovdp_ts, lags=lag)
    result = model.fit()
    if result.aic < best_aic:
        best_aic = result.aic
        best_lag = lag

ar_model = AutoReg(ovdp_ts, lags=best_lag)
ar_result = ar_model.fit()
print("ovdp Univariate AR Model - Best lag (AIC):", best_lag)
ar_result.summary()

ovdp Univariate AR Model - Best lag (AIC): 12


<class 'statsmodels.iolib.summary.Summary'>
"""
                            AutoReg Model Results                             
==============================================================================
Dep. Variable:                   ОВДП   No. Observations:                   61
Model:                    AutoReg(12)   Log Likelihood                -158.146
Method:               Conditional MLE   S.D. of innovations              6.101
Date:                Fri, 13 Feb 2026   AIC                            344.292
Time:                        12:28:16   BIC                            370.777
Sample:                    02-01-2022   HQIC                           354.340
                         - 02-01-2026                                         
==============================================================================
                 coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------
const         -4.0463      2.569     -1.575      0.115      -9.081       0.988
ОВДП.L1        0.5268      0.137      3.859      0.000       0.259       0.794
ОВДП.L2        0.0375      0.155      0.243      0.808      -0.265       0.341
ОВДП.L3        0.1991      0.155      1.288      0.198      -0.104       0.502
ОВДП.L4        0.0122      0.149      0.082      0.935      -0.280       0.305
ОВДП.L5        0.0303      0.145      0.209      0.835      -0.254       0.315
ОВДП.L6       -0.0461      0.146     -0.316      0.752      -0.332       0.240
ОВДП.L7        0.2214      0.145      1.532      0.126      -0.062       0.505
ОВДП.L8       -0.4274      0.149     -2.860      0.004      -0.720      -0.134
ОВДП.L9        0.3848      0.161      2.390      0.017       0.069       0.700
ОВДП.L10       0.0191      0.172      0.111      0.911      -0.318       0.356
ОВДП.L11       0.0623      0.191      0.325      0.745      -0.313       0.437
ОВДП.L12       0.2322      0.177      1.311      0.190      -0.115       0.579
                                    Roots                                     
==============================================================================
                   Real          Imaginary           Modulus         Frequency
------------------------------------------------------------------------------
AR.1             0.9522           -0.0000j            0.9522           -0.0000
AR.2             0.9343           -0.4922j            1.0560           -0.0772
AR.3             0.9343           +0.4922j            1.0560            0.0772
AR.4             0.5340           -1.0120j            1.1443           -0.1727
AR.5             0.5340           +1.0120j            1.1443            0.1727
AR.6             0.0866           -1.2622j            1.2652           -0.2391
AR.7             0.0866           +1.2622j            1.2652            0.2391
AR.8            -0.4291           -0.9643j            1.0555           -0.3166
AR.9            -0.4291           +0.9643j            1.0555            0.3166
AR.10           -0.9436           -0.4537j            1.0470           -0.4287
AR.11           -0.9436           +0.4537j            1.0470            0.4287
AR.12           -1.5848           -0.0000j            1.5848           -0.5000
------------------------------------------------------------------------------
"""

This model shows that Google Trends interest in “ovdp” is strongly persistent. The high lag order (12 lags) and the significant L1, L8, and L9 coefficients suggest that search interest tends to persist month to month (L1) and also has a longer cyclical component around 8–9 months. So interest in ovdp is not short-lived; past levels predict future levels for up to about a year, which fits a topic that returns to attention periodically rather than disappearing quickly.

In [7]:
investitsii_url = "https://raw.githubusercontent.com/Tleuberlin/group4-project2-domain.github.io/main/datasets/investitsii_ts_5Y.csv"
investitsii = pd.read_csv(investitsii_url)
investitsii["Time"] = pd.to_datetime(investitsii["Time"])

In [8]:
ovdp_investitsii = ovdp.merge(investitsii, on="Time")
ovdp_investitsii = ovdp_investitsii.set_index("Time")[["ОВДП", "інвестиції"]].astype(float)

var_oi = VAR(ovdp_investitsii)
lag_order_oi = var_oi.select_order(maxlags=12)
best_lag_oi = lag_order_oi.selected_orders["aic"]

var_model_oi = VAR(ovdp_investitsii)
var_result_oi = var_model_oi.fit(best_lag_oi)
print("VAR OVDP + Investitsii - Best lag (AIC):", best_lag_oi)
var_result_oi.summary()

VAR OVDP + Investitsii - Best lag (AIC): 3


  Summary of Regression Results   
Model:                         VAR
Method:                        OLS
Date:           Fri, 13, Feb, 2026
Time:                     12:28:20
--------------------------------------------------------------------
No. of Equations:         2.00000    BIC:                    9.51629
Nobs:                     58.0000    HQIC:                   9.21266
Log likelihood:          -412.146    FPE:                    8277.55
AIC:                      9.01894    Det(Omega_mle):         6590.69
--------------------------------------------------------------------
Results for equation ОВДП
                   coefficient       std. error           t-stat            prob
--------------------------------------------------------------------------------
const                -5.126572         5.183578           -0.989           0.323
L1.ОВДП               0.403495         0.142877            2.824           0.005
L1.інвестиції         0.088874         0.090507            0.

This model says that, for Google Trends data on “ОВДП” and “інвестиції” , the two series mainly move on their own past, not on each other. OVDP search interest is driven by its own lags; investment search interest is strongly driven by its own L1 (and a negative L2) and is not significantly predicted by OVDP searches. The 0.34 residual correlation implies they are affected by some common shocks (most likely - news about war and economic views ), but there is no clear lead–lag relationship between searches for OVDP and searches for investments.